# 02 · CPCB National AQI engine — O(1) sub-index lookup & daily India AQI map

**BAH 2026 PS3 · Objective-1, Stage 3 (deterministic AQI computation).**

AQI is **not** learned — it is a deterministic, O(1)-per-cell CPCB-NAQI lookup applied on top
of (predicted) surface pollutant concentrations. This notebook shows the pure, vectorized
engine in `aqi_india.aqi`: the piecewise-linear sub-index via `np.searchsorted`, the
max-of-sub-index aggregation with the **≥3-pollutant AND (PM2.5|PM10)** validity rule, the
responsible-pollutant argmax, golden worked examples, and a rendered daily India AQI map.

In [ ]:
import sys, pathlib
# Make the src/ layout importable when running from the notebooks/ folder
# without an editable install. If aqi_india is already installed this is a no-op.
_repo = pathlib.Path.cwd()
for _ in range(4):
    if (_repo / 'src' / 'aqi_india').is_dir():
        sys.path.insert(0, str(_repo / 'src'))
        break
    _repo = _repo.parent
import aqi_india
print('aqi_india', aqi_india.__version__)

## 1. The frozen CPCB breakpoint tables

`aqi_india.aqi.breakpoints` is the single source of truth: the per-pollutant breakpoint
edges, the six AQI band lo/hi values, the category names/colours, and the validity constants
(`MIN_VALID_POLLUTANTS = 3`, PM requirement). Units: CO in `mg/m3`, all others `ug/m3`.

In [ ]:
import numpy as np, pandas as pd
from aqi_india.aqi import breakpoints as bp

print('POLLUTANTS          :', bp.POLLUTANTS)
print('PM_POLLUTANTS       :', bp.PM_POLLUTANTS)
print('MIN_VALID_POLLUTANTS:', bp.MIN_VALID_POLLUTANTS)
print('AQI band lo / hi    :', bp.AQI_BAND_LO, '/', bp.AQI_BAND_HI)
print('Category names      :', bp.CATEGORY_NAMES)
print('Averaging hours     :', bp.AVERAGING_HOURS)

# The breakpoint edges (7 per pollutant: lower edges of the 6 bands + the top cap).
pd.DataFrame({p: bp.BREAKPOINTS[p] for p in bp.POLLUTANTS})

## 2. The O(1) sub-index lookup

`aqi_india.aqi.naqi.sub_index(conc, pollutant)` finds the breakpoint segment with a single
`searchsorted` and applies the CPCB piecewise-linear formula
$I_p = \frac{I_{Hi}-I_{Lo}}{BP_{Hi}-BP_{Lo}}(C_p - BP_{Lo}) + I_{Lo}$,
clamped to `[0, 500]`. It is fully vectorized and NaN/negative-safe. Below we sweep PM2.5 to
show the characteristic piecewise-linear curve with kinks at the CPCB breakpoints.

In [ ]:
from aqi_india.aqi.naqi import sub_index
import matplotlib.pyplot as plt

pm25_sweep = np.linspace(0, 320, 400)
si = sub_index(pm25_sweep, 'pm25')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(pm25_sweep, si, lw=2)
for edge in bp.BREAKPOINTS['pm25'][1:6]:
    ax.axvline(edge, color='0.7', ls=':', lw=1)
for band in [50, 100, 200, 300, 400]:
    ax.axhline(band, color='0.85', ls=':', lw=1)
ax.set_xlabel('PM2.5 (ug/m3)'); ax.set_ylabel('PM2.5 sub-index')
ax.set_title('CPCB sub-index is piecewise-linear with kinks at the breakpoints')
plt.show()

## 3. Golden worked examples

A few hand-checkable cases. Note PM2.5 = 90 lands exactly on a breakpoint and maps to **200**
(the lower segment that *ends* at the breakpoint), matching the CPCB worked examples.

In [ ]:
golden = {
    'pm25=30  -> 50  (top of Good)':            ('pm25', 30.0),
    'pm25=90  -> 200 (breakpoint, lower band)': ('pm25', 90.0),
    'pm25=250 -> 400 (top of Very Poor)':       ('pm25', 250.0),
    'no2=80   -> 100 (top of Satisfactory)':    ('no2', 80.0),
    'co=10 mg/m3 -> 100':                        ('co', 10.0),
    'o3=100   -> 100':                           ('o3', 100.0),
}
for label, (pol, conc) in golden.items():
    print(f'{label:42s}  sub_index({pol}={conc}) = {float(sub_index(conc, pol)):.1f}')

## 4. The validity rule + responsible pollutant

`aqi_from_subindices` takes the **max** of the available sub-indices, but only where a cell
has `>= 3` pollutants **and** at least one of PM2.5/PM10 — otherwise the AQI is `NaN`. It can
also return the **responsible pollutant** (the argmax). The DataFrame helper `compute_aqi`
applies the whole thing column-wise and adds `aqi`, `aqi_category`, `aqi_responsible`.

In [ ]:
from aqi_india.aqi.naqi import aqi_from_subindices, compute_aqi

demo = pd.DataFrame({
    'pm25': [30.0, 90.0, 250.0, 12.0, np.nan],
    'pm10': [60.0, 180.0, 300.0, 40.0, 55.0],
    'no2':  [40.0, 85.0, 200.0, np.nan, 30.0],
    'so2':  [20.0, 60.0, 800.0, np.nan, np.nan],   # row 4 has < 3 pollutants -> invalid
    'co':   [1.0, 12.0, 20.0, np.nan, 1.5],
    'o3':   [50.0, 120.0, 180.0, np.nan, 70.0],
})
compute_aqi(demo)[['aqi', 'aqi_category', 'aqi_responsible']].join(demo)

## 5. A daily India AQI map from the synthetic surface fields

To draw a *gridded* AQI map we need gridded **surface** concentrations. The full Objective-1
model (notebook 03) predicts these; here, to demonstrate the gridded engine end-to-end, we
derive plausible surface fields from the synthetic cube using the same textbook relationships
the simulator encodes (PM ∝ AOD/BLH·f(RH); gases as sub-linear functions of their columns),
then call `aqi_india.aqi.grid.apply_naqi_grid` (a thin wrapper over `naqi.compute_aqi_grid`).

In [ ]:
import xarray as xr
from aqi_india.sim import synthetic as sim
from aqi_india.sim.synthetic import _COL_SCALE          # the simulator's column scales
from aqi_india.fusion.gapfill import fill_gaps

# A small clear demo cube, with the cloud gaps closed by the real gap-fill engine
# so the AQI map is contiguous.
g = sim.make_grid('2023-11-01', n_days=3, res=0.25, seed=7)
g = fill_gaps(g, method='auto')

aod = g['aod'] / _COL_SCALE['aod']
blh_km = (g['blh'] / 1000.0).clip(0.2, None)
frh = 1.0 / (1.0 - (g['rh'].clip(0, 95) / 100.0)) ** 0.5

surface = xr.Dataset(coords=g.coords)
surface['pm25'] = (35.0 * aod / blh_km * frh).clip(2, 900)
surface['pm10'] = (surface['pm25'] * 1.7 + 8.0).clip(2, 1200)
surface['no2'] = (25.0 * (g['no2_col'] / _COL_SCALE['no2_col']) ** 0.8).clip(0.5, 500)
surface['so2'] = (12.0 * (g['so2_col'] / _COL_SCALE['so2_col']) ** 0.7).clip(0.2, 600)
surface['co'] = (0.8 * (g['co_col'] / _COL_SCALE['co_col']) ** 0.9).clip(0.05, 50)   # mg/m3
surface['o3'] = (18.0 + 6.0 * (g['o3_col'] / _COL_SCALE['o3_col'])).clip(1, 400)
list(surface.data_vars)

In [ ]:
from aqi_india.aqi.grid import apply_naqi_grid

aqi_ds = apply_naqi_grid(surface)   # adds 'aqi', 'aqi_responsible' (code), 'aqi_category'
aqi_ds

### Render the AQI map with the official CPCB colour scale

`aqi_india.aqi.grid` also provides the categorical CPCB colour LUT. We colourise the first
day and add a legend of the six bands (Good → Severe).

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

edges = [0, 50, 100, 200, 300, 400, 500]
cmap = ListedColormap(list(bp.CATEGORY_COLORS))
norm = BoundaryNorm(edges, cmap.N)

fig, ax = plt.subplots(figsize=(8.5, 7.5))
aqi_ds['aqi'].isel(time=0).plot(ax=ax, cmap=cmap, norm=norm,
                                cbar_kwargs={'ticks': edges, 'label': 'CPCB NAQI'})
ax.set_title(f"Daily India AQI — {str(aqi_ds.time.values[0])[:10]} (synthetic surface fields)")
ax.set_xlabel('lon'); ax.set_ylabel('lat'); ax.set_aspect('equal')
legend = [Patch(facecolor=c, label=n) for c, n in zip(bp.CATEGORY_COLORS, bp.CATEGORY_NAMES)]
ax.legend(handles=legend, loc='lower left', fontsize=8, title='CPCB band')
plt.show()

### Which pollutant drives the AQI where?

The integer `aqi_responsible` grid decodes to pollutant names via its
`pollutant_codes` attribute — useful for the public-health message ("today's AQI is driven by
PM2.5 over the IGP").

In [ ]:
codes = aqi_ds['aqi_responsible'].attrs['pollutant_codes']
print('responsible-pollutant code map:', codes)
resp0 = aqi_ds['aqi_responsible'].isel(time=0)
fig, ax = plt.subplots(figsize=(8, 7))
im = resp0.plot(ax=ax, cmap='tab10', add_colorbar=True,
                cbar_kwargs={'label': 'responsible pollutant code (-1 = invalid)'})
ax.set_title('Responsible (max sub-index) pollutant'); ax.set_aspect('equal')
plt.show()

## Summary

The CPCB NAQI engine is a deterministic, vectorized O(1)-per-cell lookup: `searchsorted`
sub-index → max aggregation under the ≥3-pollutant + PM validity rule → responsible-pollutant
argmax. Golden examples match the CPCB worked cases, and the same engine scales cleanly to a
daily gridded India AQI map via `apply_naqi_grid`. In notebook 03 the surface fields come from
the trained model instead of the hand-derived demo here.